In [24]:
import torch
import torch.nn.functional as F
from torch import nn
import csv
import os
from glob import glob
import numpy as np
from lion_pytorch import Lion
from PIL import Image
from torch import optim
from torch.types import Tensor
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

In [26]:
""" Global parameters """
H = 512
W = 512

# Build the Model

In [27]:
def conv_block(in_ch, out_ch, rate=1):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=rate, dilation=rate),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    )


class RSUL(nn.Module):
    def __init__(self, in_ch, out_ch, int_ch, num_layers, rate=2):
        super().__init__()

        self.pool = nn.MaxPool2d(2)

        # Initial Conv
        self.init = conv_block(in_ch, out_ch)

        # Encoder
        self.encoder = nn.ModuleList()
        self.encoder.append(conv_block(out_ch, int_ch))

        for _ in range(num_layers - 2):
            self.encoder.append(conv_block(int_ch, int_ch))

        # Bridge
        self.bridge = conv_block(int_ch, int_ch, rate)

        # Decoder
        self.decoder = nn.ModuleList()

        self.decoder.append(conv_block(int_ch * 2, int_ch))

        for _ in range(num_layers - 3):
            self.decoder.append(conv_block(int_ch * 2, int_ch))

        self.final = conv_block(int_ch * 2, out_ch)

    def forward(self, inputs):
        # Initial Conv
        x = self.init(inputs)
        init_feats = x

        # Encoder
        skip = []

        x = self.encoder[0](x)
        skip.append(x)

        for i in range(len(self.encoder) - 1):
            x = self.pool(x)
            x = self.encoder[i + 1](x)
            skip.append(x)

        # Bridge
        x = self.bridge(x)

        # Decoder
        skip.reverse()

        x = torch.cat([x, skip[0]], dim=1)
        x = self.decoder[0](x)

        for i in range(len(self.decoder) - 1):
            x = F.interpolate(
                x,
                scale_factor=2,
                mode="bilinear",
                align_corners=False,
            )
            x = torch.cat([x, skip[i + 1]], dim=1)
            x = self.decoder[i + 1](x)

        x = F.interpolate(
            x,
            scale_factor=2,
            mode="bilinear",
            align_corners=False,
        )
        x = torch.cat([x, skip[-1]], dim=1)
        x = self.final(x)

        # Add
        x = x + init_feats
        return x


class RSU4F(nn.Module):
    def __init__(self, in_ch, out_ch, int_ch):
        super().__init__()

        # Initial Conv
        self.conv0 = conv_block(in_ch, out_ch, rate=1)

        # Encoder
        self.conv1 = conv_block(out_ch, int_ch, rate=1)
        self.conv2 = conv_block(int_ch, int_ch, rate=2)
        self.conv3 = conv_block(int_ch, int_ch, rate=4)

        # Bridge
        self.bridge = conv_block(int_ch, int_ch, rate=8)

        # Decoder
        self.dec3 = conv_block(int_ch * 2, int_ch, rate=4)
        self.dec2 = conv_block(int_ch * 2, int_ch, rate=2)
        self.dec1 = conv_block(int_ch * 2, out_ch, rate=1)

    def forward(self, inputs):
        """Initial Conv"""
        x0 = self.conv0(inputs)

        """ Encoder """
        x1 = self.conv1(x0)
        x2 = self.conv2(x1)
        x3 = self.conv3(x2)

        """ Bridge """
        x4 = self.bridge(x3)

        """ Decoder """
        x = torch.cat([x4, x3], dim=1)
        x = self.dec3(x)

        x = torch.cat([x, x2], dim=1)
        x = self.dec2(x)

        x = torch.cat([x, x1], dim=1)
        x = self.dec1(x)

        """ Addition """
        x = x + x0
        return x


class U2Net(nn.Module):
    def __init__(self, in_ch, out_ch, int_ch, num_classes=1):
        super().__init__()

        self.pool = nn.MaxPool2d(2)

        """ Encoder """
        self.stage1 = RSUL(in_ch, out_ch[0], int_ch[0], 7)
        self.stage2 = RSUL(out_ch[0], out_ch[1], int_ch[1], 6)
        self.stage3 = RSUL(out_ch[1], out_ch[2], int_ch[2], 5)
        self.stage4 = RSUL(out_ch[2], out_ch[3], int_ch[3], 4)
        self.stage5 = RSU4F(out_ch[3], out_ch[4], int_ch[4])

        """ Bridge """
        self.stage6 = RSU4F(out_ch[4], out_ch[5], int_ch[5])

        """ Decoder """
        self.stage5d = RSU4F(out_ch[5] + out_ch[4], out_ch[6], int_ch[6])
        self.stage4d = RSUL(out_ch[6] + out_ch[3], out_ch[7], int_ch[7], 4)
        self.stage3d = RSUL(out_ch[7] + out_ch[2], out_ch[8], int_ch[8], 5)
        self.stage2d = RSUL(out_ch[8] + out_ch[1], out_ch[9], int_ch[9], 6)
        self.stage1d = RSUL(out_ch[9] + out_ch[0], out_ch[10], int_ch[10], 7)

        """ Side Outputs """
        self.side1 = nn.Conv2d(out_ch[10], num_classes, 3, padding=1)
        self.side2 = nn.Conv2d(out_ch[9], num_classes, 3, padding=1)
        self.side3 = nn.Conv2d(out_ch[8], num_classes, 3, padding=1)
        self.side4 = nn.Conv2d(out_ch[7], num_classes, 3, padding=1)
        self.side5 = nn.Conv2d(out_ch[6], num_classes, 3, padding=1)
        self.side6 = nn.Conv2d(out_ch[5], num_classes, 3, padding=1)

        self.outconv = nn.Conv2d(num_classes * 6, num_classes, 3, padding=1)

    def forward(self, inputs):
        """Input Layer"""
        s0 = inputs

        """ Encoder """
        s1 = self.stage1(s0)
        p1 = self.pool(s1)

        s2 = self.stage2(p1)
        p2 = self.pool(s2)

        s3 = self.stage3(p2)
        p3 = self.pool(s3)

        s4 = self.stage4(p3)
        p4 = self.pool(s4)

        s5 = self.stage5(p4)
        p5 = self.pool(s5)

        """ Bridge """
        b1 = self.stage6(p5)
        b2 = F.interpolate(b1, scale_factor=2, mode="bilinear", align_corners=False)

        """ Decoder """
        d1 = torch.cat([b2, s5], dim=1)
        d1 = self.stage5d(d1)
        u1 = F.interpolate(d1, scale_factor=2, mode="bilinear", align_corners=False)

        d2 = torch.cat([u1, s4], dim=1)
        d2 = self.stage4d(d2)
        u2 = F.interpolate(d2, scale_factor=2, mode="bilinear", align_corners=False)

        d3 = torch.cat([u2, s3], dim=1)
        d3 = self.stage3d(d3)
        u3 = F.interpolate(d3, scale_factor=2, mode="bilinear", align_corners=False)

        d4 = torch.cat([u3, s2], dim=1)
        d4 = self.stage2d(d4)
        u4 = F.interpolate(d4, scale_factor=2, mode="bilinear", align_corners=False)

        d5 = torch.cat([u4, s1], dim=1)
        d5 = self.stage1d(d5)

        """ Side Outputs """
        y1 = self.side1(d5)

        y2 = self.side2(d4)
        y2 = F.interpolate(y2, scale_factor=2, mode="bilinear", align_corners=False)

        y3 = self.side3(d3)
        y3 = F.interpolate(y3, scale_factor=4, mode="bilinear", align_corners=False)

        y4 = self.side4(d2)
        y4 = F.interpolate(y4, scale_factor=8, mode="bilinear", align_corners=False)

        y5 = self.side5(d1)
        y5 = F.interpolate(y5, scale_factor=16, mode="bilinear", align_corners=False)

        y6 = self.side6(b1)
        y6 = F.interpolate(y6, scale_factor=32, mode="bilinear", align_corners=False)

        y0 = torch.cat([y1, y2, y3, y4, y5, y6], dim=1)
        y0 = self.outconv(y0)

        y0 = torch.sigmoid(y0)
        y1 = torch.sigmoid(y1)
        y2 = torch.sigmoid(y2)
        y3 = torch.sigmoid(y3)
        y4 = torch.sigmoid(y4)
        y5 = torch.sigmoid(y5)
        y6 = torch.sigmoid(y6)

        return y0, y1, y2, y3, y4, y5, y6

    def fit(self,
            epochs: int,
            optimizer,
            scheduler,
            train_dataloader: DataLoader,
            valid_dataloader: DataLoader,
            log_csv_path:str,
            device: str = "cuda" if torch.cuda.is_available else "cpu"):

        best_val_loss = float("inf")
        patience = 20
        wait = 0

        with open(log_csv_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["epoch", "train_loss", "val_loss", "lr"])

            for epoch in range(epochs):
                print(f"\nEpoch [{epoch + 1}/{epochs}]")

                # -------------------- Train --------------------
                self.train()
                train_loss = 0.0

                train_bar = tqdm(train_dataloader, desc="Training", leave=False, dynamic_ncols=True)
                for images, masks in train_bar:
                    images = images.to(device)
                    masks = masks.to(device)

                    optimizer.zero_grad()
                    outputs = model(images)
                    loss: Tensor = sum(criterion(d, masks) for d in outputs)
                    loss.backward()
                    optimizer.step()

                    train_loss += loss.item()
                    train_bar.set_postfix(loss=f"{train_loss:.4f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

                train_loss /= len(train_dataloader)

                # -------------------- Validation --------------------
                self.eval()
                val_loss = 0.0

                valid_bar = tqdm(valid_dataloader, desc="Validation", leave=False, dynamic_ncols=True)

                with torch.no_grad():
                    for images, masks in valid_bar:
                        images = images.to(device)
                        masks = masks.to(device)

                        outputs = model(images)
                        loss: Tensor = sum(criterion(d, masks) for d in outputs)
                        val_loss += loss.item()
                        valid_bar.set_postfix(loss=f"{val_loss:.4f}")

                val_loss /= len(valid_dataloader)

                scheduler.step(val_loss)

                current_lr = optimizer.param_groups[0]["lr"]
                writer.writerow([epoch + 1, train_loss, val_loss, current_lr])
                f.flush()

                # ModelCheckpoint(save_best_only=True)
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    wait = 0
                    torch.save(model.state_dict(), model_path)
                    print(f"Epoch {epoch + 1}: saved best model")
                else:
                    wait += 1

                print(
                    f"Epoch {epoch + 1}/{num_epochs} "
                    f"train_loss={train_loss:.4f} "
                    f"val_loss={val_loss:.4f} "
                    f"lr={current_lr:.2e}"
                )

                # EarlyStopping(patience=20)
                if wait >= patience:
                    print("Early stopping triggered")
                    break


def build_u2net(in_ch=3, num_classes=1):
    out_ch = [64, 128, 256, 512, 512, 512, 512, 256, 128, 64, 64]
    int_ch = [32, 32, 64, 128, 256, 256, 256, 128, 64, 32, 16]

    model = U2Net(in_ch, out_ch, int_ch, num_classes=num_classes)
    return model


def build_u2net_lite(in_ch=3, num_classes=1):
    out_ch = [64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64]
    int_ch = [16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16]
    model = U2Net(in_ch, out_ch, int_ch, num_classes=num_classes)
    return model


In [28]:
def check_vram_requirement(model: nn.Module):
    device = "cuda" if torch.cuda.is_available else "cpu"
    model = model.to(device)

    model.train()
    x = torch.randn(1, 3, 512, 512, device=device)
    torch.cuda.reset_peak_memory_stats()
    outputs = model(x)

    loss = outputs[0].mean()  # Tensor
    loss.backward()

    vram_needed_mbs = torch.cuda.max_memory_allocated() / 1024 ** 2
    print(f"Estimated Training VRAM: {vram_needed_mbs:.2f}", "MB")


model = build_u2net()

# Total and Trainable parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
check_vram_requirement(model=model)

Total Parameters: 44,009,917
Trainable Parameters: 44,009,917
Estimated Training VRAM: 2499.84 MB


# Dataset Creation

In [29]:
def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)


def load_dataset(path):
    train_x = sorted(glob(os.path.join(path, "train", "blurred_image", "*.jpg")))[:50]
    train_y = sorted(glob(os.path.join(path, "train", "mask", "*.png")))[:50]

    valid_x = sorted(glob(os.path.join(path, "validation", "P3M-500-NP", "original_image", "*.jpg")))[:50]
    valid_y = sorted(glob(os.path.join(path, "validation", "P3M-500-NP", "mask", "*.png")))[:50]

    return (train_x, train_y), (valid_x, valid_y)


class SegmentationDataset(Dataset):
    def __init__(self, X, Y):
        self.X = X
        self.Y = Y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        image = Image.open(self.X[idx]).convert("RGB").resize((W, H))
        image = np.asarray(image, dtype=np.float32) / 255.0
        image = torch.from_numpy(image).permute(2, 0, 1)

        mask = Image.open(self.Y[idx]).convert("L").resize((W, H))
        mask = np.asarray(mask, dtype=np.float32) / 255.0
        mask = torch.from_numpy(mask).unsqueeze(0)

        return image, mask

# Train the Model

In [30]:
""" Seeding """
np.random.seed(3407)
torch.random.manual_seed(3407)
torch.manual_seed(3407)

device = "cuda" if torch.cuda.is_available else "cpu"
BASE_DATA_DIR = os.path.join("data")

""" Directory for storing files """
create_dir(os.path.join(BASE_DATA_DIR, "files"))

In [31]:
""" Hyperparameters """
batch_size = 2
lr = 1e-4
num_epochs = 500
model_path = os.path.join(BASE_DATA_DIR, "files", "model.pth")
csv_path = os.path.join(BASE_DATA_DIR, "files", "log.csv")

""" Dataset """
dataset_path = os.path.join(BASE_DATA_DIR, "P3M-10k")
(train_x, train_y), (valid_x, valid_y) = load_dataset(dataset_path)

print(f"Train: {len(train_x)} - {len(train_y)}")
print(f"Valid: {len(valid_x)} - {len(valid_y)}")

train_dataloader = DataLoader(SegmentationDataset(train_x, train_y), batch_size=batch_size, shuffle=False)
valid_dataloader = DataLoader(SegmentationDataset(valid_x, valid_y), batch_size=batch_size, shuffle=False)

""" Model """
model = build_u2net_lite()
model.to(device)
# model.load_state_dict(torch.load(model_path))

criterion = torch.nn.BCEWithLogitsLoss()
optimizer = Lion(model.parameters(), lr=lr, weight_decay=lr / 2)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.1,
    patience=5,
    min_lr=1e-7,
)

model.fit(
    epochs=num_epochs,
    optimizer=optimizer,
    scheduler=scheduler,
    train_dataloader=train_dataloader,
    valid_dataloader=valid_dataloader,
    log_csv_path=csv_path,
    device=device
)

Train: 50 - 50
Valid: 50 - 50

Epoch [1/500]


KeyboardInterrupt: 

# Test the model

In [36]:
""" Directory for storing files """
for dirname in ["joint", "mask"]:
    create_dir(os.path.join(BASE_DATA_DIR, "results", dirname))

""" Load the model """
model_path = os.path.join(BASE_DATA_DIR, "files", "model.pth")
model = build_u2net_lite()
model.to(device)
model.load_state_dict(torch.load(model_path, weights_only=True))

""" Dataset """
images = glob(os.path.join(BASE_DATA_DIR, "test/*"))
print(f"Images: {len(images)}")


""" Prediction """
for x in tqdm(images, total=len(images)):
    """ Extracting the name """
    name = x.split("/")[-1]

    """ Reading the image """
    image = Image.open(x).convert("RGB").resize((W, H))
    image = np.asarray(image, dtype=np.float32) / 255.0
    image = torch.from_numpy(image).permute(2, 0, 1).unsqueeze(0)
    image = image.to(device)

    """ Prediction """
    pred = model(image)
    pred = [item.detach().cpu().numpy() for item in pred]

    line = np.ones((H, 10, 3), dtype=np.uint8) * 255

    """ Joint and save mask """
    pred_list = []
    for item in pred:
        p = (item.squeeze() * 255).astype(np.uint8)
        p = np.stack([p, p, p], axis=-1)

        pred_list.append(p)
        pred_list.append(line)

    save_image_path = os.path.join(BASE_DATA_DIR, "results", "mask", name)
    cat_images = np.concatenate(pred_list, axis=1)
    Image.fromarray(cat_images).save(save_image_path)

    """ Save final mask """
    image = Image.open(x)
    image_w, image_h = image.size

    prediction = pred[0].squeeze()
    mask = Image.fromarray((prediction * 255).astype(np.uint8))
    mask = mask.resize((image_w, image_h), Image.Resampling.BILINEAR)
    y0 = np.array(mask, dtype=np.float32) / 255.0
    y0 = np.stack([y0, y0, y0], axis=-1)

    line = np.ones((image_h, 10, 3), dtype=np.uint8) * 255

    cat_images = np.concatenate([
        np.asarray(image).astype(np.uint8),
        line,
        (y0 * 255).astype(np.uint8),
        line,
        (np.asarray(image) * y0).astype(np.uint8),
    ], axis=1)
    save_image_path = os.path.join(BASE_DATA_DIR, "results", "joint", name)
    Image.fromarray(cat_images).save(save_image_path)


Images: 19


 63%|██████▎   | 12/19 [00:07<00:04,  1.63it/s]


KeyboardInterrupt: 